In [2]:
import polars as pl
from pathlib import Path

from insightfuel_data_platform.pipelines.anp import processar_particoes_anp

from insightfuel_data_platform.storage.files import descobrir_csvs

pasta_bronze = Path("../data/bronze/anp/automotivos")
pasta_silver = Path("../data/silver/anp/automotivos")

processados = processar_particoes_anp(
    pasta_bronze,
    pasta_silver,
)

for caminho in processados:
    print(caminho)

In [1]:
from pathlib import Path
import polars as pl

caminho = Path(
    "../data/silver/anp/automotivos/"
    "ano=2024/semestre=1/dados.parquet"
)

df = pl.read_parquet(caminho)

df = df.with_columns(
    pl.col("data_coleta")
    .dt.truncate("1mo")
    .alias("ano_mes")
)

gold_teste = (
    df
    .group_by([
        "ano_mes",
        "uf",
        "municipio",
        "produto"
    ])
    .agg(
        pl.col("valor_venda").mean().alias("preco_medio"),
        pl.col("valor_venda").median().alias("preco_mediano"),
        pl.col("valor_venda").min().alias("preco_minimo"),
        pl.col("valor_venda").max().alias("preco_maximo"),
        pl.col("valor_venda").std().alias("desvio_padrao"),
        pl.len().alias("qtd_coletas"),
    )
    .sort([
        "ano_mes",
        "uf",
        "municipio",
        "produto"
    ])
)

print(gold_teste.shape)
print(gold_teste.schema)

gold_teste.head(10)

print("Linhas Silver:", df.height)
print("Linhas Gold:", gold_teste.height)

(14466, 10)
Schema({'ano_mes': Date, 'uf': String, 'municipio': String, 'produto': String, 'preco_medio': Float64, 'preco_mediano': Float64, 'preco_minimo': Decimal(precision=10, scale=2), 'preco_maximo': Decimal(precision=10, scale=2), 'desvio_padrao': Float64, 'qtd_coletas': UInt32})
Linhas Silver: 477144
Linhas Gold: 14466
